# 第 12 章 · 知识蒸馏

> 📄 配套动画网页：`../12_distill.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

让一个小模型去模仿一个大模型。关键是：老师输出的完整概率分布里藏着「暗知识」。

---


## 🎯 本章目标

- 理解黑盒蒸馏与白盒蒸馏的区别
- 掌握 `F.kl_div`、`F.log_softmax` 与温度缩放
- 理解为什么 KL 里 input 传 log-prob、target 传概率
- 能实现组合了 CE 与 KL 的蒸馏损失

---


## 🖼 官方配图

![学生模型就是这个架构；老师通常是一个结构相同但规模更大的模型，用它的输出分布来指导学生](../images/LLM-structure.jpg)

*学生模型就是这个架构；老师通常是一个结构相同但规模更大的模型，用它的输出分布来指导学生*

---


## 📖 背景：软标签里的暗知识

假设正确答案是「猫」。硬标签只说「猫=1，其它=0」。
但老师的软标签可能是「猫 0.7，狗 0.2，兔子 0.05」——
这告诉学生**「狗比兔子更接近猫」**。这种相似性信息是硬标签完全没有的。

```python
loss = α · CE(student_logits, hard_label)
     + (1-α) · T² · KL( softmax(teacher/T) ‖ softmax(student/T) )
```

乘 `T²` 是为了补偿温度把梯度变小的问题。温度 `T > 1` 会把分布摊平，让暗知识更明显。

---



# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 12.1 温度缩放：`logits / T`

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([[3.2, 2.1, 1.4, 0.2, -0.6, -1.1]])

for T in [1.0, 2.0, 4.0]:
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T}: {[round(x, 3) for x in p.squeeze().tolist()]}")
print("\n观察：T 越大分布越平，低概率词越明显")

### 12.2 `F.log_softmax`：更稳定的 log 概率

In [ ]:
x = torch.tensor([[3.2, 2.1, 1.4]])
print("log_softmax  :", F.log_softmax(x, dim=-1))
print("log(softmax) :", torch.log(F.softmax(x, dim=-1)))
print("两者一致:", torch.allclose(F.log_softmax(x, -1), torch.log(F.softmax(x, -1)), atol=1e-6))

### 12.3 `F.kl_div` 的参数约定（最容易错的地方）

`F.kl_div(input, target)` 计算的是 `KL(target ‖ exp(input))`。

- `input`：**log 概率**（用 `log_softmax`）
- `target`：**概率**（用 `softmax`）

```python
kl = F.kl_div(F.log_softmax(student/T, -1), F.softmax(teacher/T, -1), reduction="batchmean")
```

In [ ]:
student = torch.tensor([[2.4, 1.6, 0.9, 0.5, -0.2, -0.5]])
teacher = torch.tensor([[3.2, 2.1, 1.4, 0.2, -0.6, -1.1]])
T = 2.0

kl = F.kl_div(F.log_softmax(student / T, dim=-1),
              F.softmax(teacher / T, dim=-1),
              reduction="batchmean") * (T ** 2)
print("KL(teacher ‖ student) =", kl.item())

# 手工验证
p = F.softmax(teacher / T, -1)
q = F.softmax(student / T, -1)
manual = (p * (p.log() - q.log())).sum()
print("手工计算            =", manual.item())

### 12.4 组合损失：CE + KL

In [ ]:
def distill_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    ce = F.cross_entropy(student_logits, labels, ignore_index=-100)
    kl = F.kl_div(F.log_softmax(student_logits / T, -1),
                  F.softmax(teacher_logits / T, -1),
                  reduction="batchmean") * (T ** 2)
    return alpha * ce + (1 - alpha) * kl, ce.item(), kl.item()

s = torch.randn(4, 100)
t = torch.randn(4, 100)
y = torch.randint(0, 100, (4,))
loss, ce, kl = distill_loss(s, t, y)
print(f"总损失={loss.item():.4f}  CE={ce:.4f}  KL={kl:.4f}")

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

现在自己实现完整的蒸馏损失。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 12.5 实现 `distillation_loss`

**要求**：
1. 算硬标签交叉熵：`F.cross_entropy(student_logits, labels, ignore_index=-100)`
2. 算软标签 KL：
   - `F.log_softmax(student_logits / T, dim=-1)` 作为 input
   - `F.softmax(teacher_logits / T, dim=-1)` 作为 target
   - `reduction="batchmean"`，再乘 `T ** 2`
3. 返回 `alpha * ce + (1 - alpha) * kl`

**验证**：当学生 logits 完全等于老师时，KL 应该接近 0。

In [ ]:
# ======== 你的实现：distillation_loss ========
#
# 【要求】
#   - 签名：distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5)
#   - ce = F.cross_entropy(student_logits, labels, ignore_index=-100)
#   - kl = F.kl_div(log_softmax(student/T), softmax(teacher/T), reduction='batchmean') * T**2
#   - return alpha * ce + (1 - alpha) * kl
#   - （可选）同时返回 ce 和 kl 的数值方便观察
#
# 【可用到的函数 / 方法】
#   - `F.cross_entropy`、`F.kl_div`
#   - `F.log_softmax`、`F.softmax`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：distillation_loss ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    s = torch.randn(4, 50)
    t = torch.randn(4, 50)
    y = torch.randint(0, 50, (4,))
    out = distillation_loss(s, t, y, T=2.0, alpha=0.5)
    loss = out[0] if isinstance(out, tuple) else out
    assert torch.isfinite(loss) and loss.dim() == 0, f"应返回有限标量，得到 {loss}"

    ce = F.cross_entropy(s, y, ignore_index=-100)
    out2 = distillation_loss(s, s.clone(), y, T=2.0, alpha=0.5)
    loss2 = out2[0] if isinstance(out2, tuple) else out2
    close(loss2.item(), (0.5 * ce).item(), tol=1e-4,
          msg="学生与老师完全一致时 KL 应为 0，loss 应等于 alpha * CE")

check("distillation_loss 数值 / 学生=老师时 KL 归零", _t)

### 12.6 实现「黑盒蒸馏」的数据构造

黑盒蒸馏不需要老师的 logits，只需要老师生成的文本。写一个函数：
给定一批 prompt，用老师模型生成回答，组装成 SFT 格式的数据。

（没有真实老师模型时，用假数据模拟即可，重点是理解流程。）

In [ ]:
# ======== 你的实现：build_blackbox_data ========
#
# 【要求】
#   - 签名：build_blackbox_data(prompts, teacher, tokenizer)
#   - 对每个 prompt 调用 teacher.generate(...)，再用 tokenizer.decode 转成文本
#   - 把 (prompt, teacher_output) 组成 {'conversations': [{'role': 'user', ...}, {'role': 'assistant', ...}]}
#   - 返回与 prompts 等长的样本列表（每项都含 'conversations'）
#
# 【可用到的函数 / 方法】
#   - `teacher.generate`
#   - `tokenizer.decode`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：build_blackbox_data ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    class FakeTeacher:
        def generate(self, *a, **kw):
            return torch.zeros(1, 1, dtype=torch.long)
    class FakeTok:
        def apply_chat_template(self, convs, tokenize=False, add_generation_prompt=True):
            return "PROMPT:" + convs[-1]["content"]
        def decode(self, ids, skip_special_tokens=True):
            return "老师的回答"

    data = build_blackbox_data(["1+1=?"], FakeTeacher(), FakeTok())
    assert isinstance(data, list) and len(data) == 1, f"应返回与 prompts 等长的列表，得到 {type(data)}"
    sample = data[0]
    assert "conversations" in sample, f"每个样本应有 conversations 字段，得到 {list(sample.keys())}"
    roles = [m["role"] for m in sample["conversations"]]
    assert "assistant" in roles, f"样本里应包含 assistant 回复，得到 {roles}"

check("build_blackbox_data 结构", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能说清「暗知识」到底指什么
- [ ] 温度 T 增大时分布怎么变？
- [ ] KL 里为什么 input 是 log 概率、target 是概率？
- [ ] T² 缩放为什么必要？
- [ ] 学生等于老师时 KL 接近 0 吗？

**思考题**：蒸馏和 SFT 的区别到底在哪？如果老师的输出本身就是错的，蒸馏会怎样？
